# Deep Learning Quantization Demo

This notebook demonstrates the complete quantization pipeline:
1. Train a CNN in FP32
2. Apply Post-Training Quantization (PTQ) with calibration
3. Apply Quantization-Aware Training (QAT)
4. Compare all models: size, latency, accuracy, SQNR
5. Visualize weight distributions and error metrics

**All hyperparameters are loaded from `config.yaml`.**

In [ ]:
# Setup: add project root to path
import sys
import pathlib

PROJ_ROOT = pathlib.Path("../..")
sys.path.insert(0, str(PROJ_ROOT.resolve()))

print(f"Project root: {PROJ_ROOT.resolve()}")

In [ ]:
# Imports
import copy
import time
import logging

import matplotlib.pyplot as plt
import matplotlib
import numpy as np
import torch
import torch.nn as nn

from src.utils import (
    load_config, setup_logging, build_dataloaders,
    QuantizableLeNetCNN, get_model_size_mb,
    train_one_epoch, evaluate_accuracy, ensure_output_dir
)

# Load config
cfg = load_config(str(PROJ_ROOT / "config.yaml"))
setup_logging(cfg)

print("Config loaded.")
print(f"  PTQ backend: {cfg['ptq']['backend']}")
print(f"  QAT epochs: {cfg['qat']['epochs']}")
print(f"  Calibration percentile: {cfg['calibration']['percentile']}")

## 1. Generate Synthetic Data and Train FP32 Model

In [ ]:
# Build data loaders
train_loader, calib_loader, test_loader = build_dataloaders(cfg)

model_cfg = cfg["model"]
device = torch.device("cpu")

# Inspect data shapes
images, labels = next(iter(train_loader))
print(f"Batch shape: {images.shape}  (N, C, H, W)")
print(f"Label shape: {labels.shape}")
print(f"Train batches: {len(train_loader)} | Calib batches: {len(calib_loader)} | Test batches: {len(test_loader)}")

In [ ]:
# Build and train FP32 model
fp32_model = QuantizableLeNetCNN(
    in_channels=model_cfg["in_channels"],
    num_classes=model_cfg["num_classes"],
    hidden_dim=model_cfg["hidden_dim"],
).to(device)

print(f"Model parameters: {sum(p.numel() for p in fp32_model.parameters()):,}")
print(f"Model size: {get_model_size_mb(fp32_model):.3f} MB")

optimizer = torch.optim.Adam(fp32_model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

train_losses = []
test_accs = []

for epoch in range(1, 4):
    loss = train_one_epoch(fp32_model, train_loader, optimizer, criterion, device)
    acc = evaluate_accuracy(fp32_model, test_loader, device)
    train_losses.append(loss)
    test_accs.append(acc)
    print(f"Epoch {epoch}/3 | loss={loss:.4f} | test_acc={acc:.4f}")

fp32_acc = test_accs[-1]
fp32_size = get_model_size_mb(fp32_model)
print(f"\nFP32 final accuracy: {fp32_acc:.4f}  Size: {fp32_size:.3f} MB")

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(range(1, 4), train_losses, 'b-o', linewidth=2, markersize=8)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].grid(True, alpha=0.3)

axes[1].plot(range(1, 4), test_accs, 'g-o', linewidth=2, markersize=8)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Test Accuracy')
axes[1].grid(True, alpha=0.3)

plt.suptitle('FP32 LeNet Training Curves (Synthetic Data)', fontsize=13)
plt.tight_layout()
plt.show()

## 2. Weight Distribution: FP32 Weights

In [ ]:
# Visualize weight distributions for each layer
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle('FP32 Weight Distributions', fontsize=13, fontweight='bold')

layer_names = []
weights_data = []

for name, module in fp32_model.named_modules():
    if hasattr(module, 'weight') and module.weight is not None and not module.weight.is_quantized:
        w = module.weight.detach().cpu().flatten().numpy()
        if len(w) > 1:
            layer_names.append(name)
            weights_data.append(w)

for i, (name, w) in enumerate(zip(layer_names[:4], weights_data[:4])):
    ax = axes[i]
    ax.hist(w, bins=60, color='steelblue', alpha=0.8, edgecolor='none')
    ax.set_title(f'{name}\n({len(w):,} params)', fontsize=9)
    ax.set_xlabel('Weight value')
    stats_text = f'μ={w.mean():.3f}\nσ={w.std():.3f}\n[{w.min():.3f}, {w.max():.3f}]'
    ax.text(0.97, 0.95, stats_text, transform=ax.transAxes, ha='right', va='top',
            fontsize=8, bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

## 3. Static Post-Training Quantization (PTQ)

In [ ]:
# Apply static PTQ
backend = cfg["ptq"]["backend"]
torch.backends.quantized.engine = backend
print(f"Quantization backend: {backend}")

# Step 1: Deep copy and fuse
ptq_model = copy.deepcopy(fp32_model)
ptq_model.eval()
ptq_model.fuse_modules()
print("Conv-BN-ReLU modules fused")

# Step 2: Set qconfig
ptq_model.qconfig = torch.quantization.get_default_qconfig(backend)
print(f"QConfig set: {ptq_model.qconfig}")

# Step 3: Prepare (insert observers)
torch.quantization.prepare(ptq_model, inplace=True)
print("Observer hooks inserted")

# Step 4: Calibrate
print(f"Running calibration ({len(calib_loader)} batches)...")
with torch.no_grad():
    for images, _ in calib_loader:
        ptq_model(images)
print("Calibration complete")

# Step 5: Convert
torch.quantization.convert(ptq_model, inplace=True)
print("Model converted to INT8")

ptq_acc = evaluate_accuracy(ptq_model, test_loader, device)
ptq_size = get_model_size_mb(ptq_model)
print(f"\nPTQ INT8 accuracy: {ptq_acc:.4f} (drop: {fp32_acc - ptq_acc:.4f})")
print(f"PTQ INT8 size: {ptq_size:.3f} MB (compression: {fp32_size/ptq_size:.1f}x)")

## 4. Quantization-Aware Training (QAT)

In [ ]:
# Apply QAT
qat_backend = cfg["qat"]["backend"]
torch.backends.quantized.engine = qat_backend

qat_model = copy.deepcopy(fp32_model)
qat_model.train()
qat_model.fuse_modules()
qat_model.qconfig = torch.quantization.get_default_qat_qconfig(qat_backend)
torch.quantization.prepare_qat(qat_model, inplace=True)
print("QAT model prepared with FakeQuantize nodes")

qat_optimizer = torch.optim.Adam(
    qat_model.parameters(),
    lr=cfg["qat"]["learning_rate"],
    weight_decay=cfg["qat"]["weight_decay"]
)

qat_losses = []
qat_accs = []

for epoch in range(1, cfg["qat"]["epochs"] + 1):
    loss = train_one_epoch(qat_model, train_loader, qat_optimizer, criterion, device)
    acc = evaluate_accuracy(qat_model, test_loader, device)
    qat_losses.append(loss)
    qat_accs.append(acc)
    print(f"QAT Epoch {epoch}/{cfg['qat']['epochs']} | loss={loss:.4f} | acc={acc:.4f} (FP32 mode, simulated INT8)")

# Convert to INT8
qat_model.eval()
torch.quantization.convert(qat_model, inplace=True)

qat_acc = evaluate_accuracy(qat_model, test_loader, device)
qat_size = get_model_size_mb(qat_model)
print(f"\nQAT INT8 accuracy: {qat_acc:.4f} (drop: {fp32_acc - qat_acc:.4f})")
print(f"QAT INT8 size: {qat_size:.3f} MB (compression: {fp32_size/qat_size:.1f}x)")

## 5. Weight Distribution: FP32 vs Dequantized INT8

In [ ]:
# Compare weight distributions before and after quantization
def get_fp32_weights(model):
    """Extract all FP32 weight tensors."""
    weights = {}
    for name, module in model.named_modules():
        if hasattr(module, 'weight') and module.weight is not None:
            w = module.weight
            if w.is_quantized:
                w = w.dequantize()
            weights[name] = w.detach().cpu().float().flatten().numpy()
    return {k: v for k, v in weights.items() if len(v) > 1}

fp32_weights = get_fp32_weights(fp32_model)
ptq_weights = get_fp32_weights(ptq_model)

# Find common layers with weight data
common = [k for k in fp32_weights if k in ptq_weights][:3]

fig, axes = plt.subplots(len(common), 3, figsize=(15, 4 * len(common)))
if len(common) == 1:
    axes = [axes]

fig.suptitle('Weight Distribution: FP32 vs Dequantized INT8 (PTQ)', fontsize=13, fontweight='bold')

for row, name in enumerate(common):
    fp32_w = fp32_weights[name]
    ptq_w = ptq_weights[name]
    error = fp32_w - ptq_w

    axes[row][0].hist(fp32_w, bins=80, color='steelblue', alpha=0.8, edgecolor='none')
    axes[row][0].set_title(f'FP32 — {name}', fontsize=9)
    axes[row][0].set_xlabel('Weight value')

    axes[row][1].hist(ptq_w, bins=80, color='darkorange', alpha=0.8, edgecolor='none')
    axes[row][1].set_title(f'INT8 (dequantized) — {name}', fontsize=9)
    axes[row][1].set_xlabel('Weight value')

    mse = np.mean(error**2)
    signal_power = np.mean(fp32_w**2)
    sqnr = 10 * np.log10(signal_power / max(mse, 1e-15))
    axes[row][2].hist(error, bins=80, color='firebrick', alpha=0.8, edgecolor='none')
    axes[row][2].set_title(f'Error (FP32 - INT8) — {name}', fontsize=9)
    axes[row][2].set_xlabel('Error value')
    axes[row][2].text(0.98, 0.95, f'MSE={mse:.2e}\nSQNR={sqnr:.1f} dB',
                      transform=axes[row][2].transAxes, ha='right', va='top', fontsize=9,
                      bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.show()

## 6. Calibration Range Visualization

In [ ]:
from src.calibration.calibrators import (
    MinMaxCalibrator, PercentileCalibrator, MSECalibrator,
    KLDivergenceCalibrator, MovingAverageCalibrator
)

# Generate synthetic activation distribution (typical post-conv, with outliers)
torch.manual_seed(42)
activations = torch.cat([
    torch.randn(5000) * 0.4,
    torch.tensor([-2.8, 3.2, -3.5, 3.9])  # outliers
])

calib_cfg = cfg["calibration"]

calibrators = {
    "Min-Max": MinMaxCalibrator(),
    "Percentile": PercentileCalibrator(percentile=calib_cfg["percentile"]),
    "MSE": MSECalibrator(search_steps=calib_cfg["mse_search_steps"],
                         alpha_range=tuple(calib_cfg["mse_alpha_range"])),
    "KL-Div": KLDivergenceCalibrator(kl_bins=calib_cfg["kl_bins"],
                                     num_quantized_bins=calib_cfg["kl_num_quantized_bins"]),
    "EMA": MovingAverageCalibrator(alpha=calib_cfg["moving_average_constant"]),
}

ranges = {}
for name, cal in calibrators.items():
    cal.collect(activations)
    min_v, max_v = cal.compute_range()
    scale, zp = cal.compute_scale_zero_point(symmetric=True)
    # Compute MSE for this range
    vals = activations.numpy()
    q_max = 127
    s = max(abs(min_v), abs(max_v)) / q_max
    clipped = np.clip(vals, min_v, max_v)
    quantized = np.round(clipped / s) * s
    mse = float(np.mean((vals - quantized)**2))
    ranges[name] = {"min": min_v, "max": max_v, "scale": scale, "mse": mse}

# Plot
colors = ["steelblue", "darkorange", "firebrick", "green", "purple"]
vals_np = activations.numpy()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: histogram with range markers
ax = axes[0]
ax.hist(vals_np, bins=100, color='lightgray', alpha=0.9, edgecolor='none', label='Activations')
for (name, r), color in zip(ranges.items(), colors):
    ax.axvline(r["min"], color=color, linewidth=2, linestyle='--', alpha=0.9)
    ax.axvline(r["max"], color=color, linewidth=2, linestyle='--', alpha=0.9,
               label=f"{name}: [{r['min']:.2f}, {r['max']:.2f}]")
ax.set_xlabel('Activation value')
ax.set_title('Calibration Ranges Overlaid on Activation Distribution')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.2)

# Right: MSE per method
ax2 = axes[1]
names_list = list(ranges.keys())
mse_list = [ranges[n]["mse"] for n in names_list]
bars = ax2.bar(names_list, mse_list, color=colors, alpha=0.8)
for bar, val in zip(bars, mse_list):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height()*1.05,
             f'{val:.2e}', ha='center', fontsize=9)
ax2.set_ylabel('MSE (lower is better)')
ax2.set_title('Quantization MSE per Calibration Method')
ax2.grid(True, alpha=0.3, axis='y')

plt.suptitle('Calibration Method Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nCalibration Results Table:")
print(f"{'Method':<15} | {'Min':>8} | {'Max':>8} | {'Scale':>10} | {'MSE':>12}")
print("-" * 60)
for name, r in ranges.items():
    print(f"{name:<15} | {r['min']:>8.4f} | {r['max']:>8.4f} | {r['scale']:>10.6f} | {r['mse']:>12.2e}")

## 7. Inference Latency Benchmark

In [ ]:
# Benchmark inference latency
def benchmark_model(model, n_iterations=200, warmup=20, label="model"):
    model.eval()
    dummy = torch.randn(1, model_cfg["in_channels"], model_cfg["input_height"], model_cfg["input_width"])

    # Warmup
    with torch.no_grad():
        for _ in range(warmup):
            model(dummy)

    # Timed
    latencies = []
    with torch.no_grad():
        for _ in range(n_iterations):
            t0 = time.perf_counter()
            model(dummy)
            t1 = time.perf_counter()
            latencies.append((t1 - t0) * 1000)  # ms

    arr = np.array(latencies)
    return {
        "label": label,
        "p50": np.percentile(arr, 50),
        "p95": np.percentile(arr, 95),
        "mean": arr.mean(),
        "std": arr.std(),
        "throughput": 1000.0 / arr.mean()
    }

bench_fp32 = benchmark_model(fp32_model, label="FP32")
bench_ptq = benchmark_model(ptq_model, label="PTQ-INT8")
bench_qat = benchmark_model(qat_model, label="QAT-INT8")

print(f"{'Model':<15} | {'p50 (ms)':>10} | {'p95 (ms)':>10} | {'Throughput':>15} | {'Size (MB)':>10}")
print("-" * 70)
for bench, size in [
    (bench_fp32, fp32_size), (bench_ptq, ptq_size), (bench_qat, qat_size)
]:
    print(f"{bench['label']:<15} | {bench['p50']:>10.3f} | {bench['p95']:>10.3f} | {bench['throughput']:>15.1f} | {size:>10.3f}")

## 8. Comprehensive Comparison Dashboard

In [ ]:
# Full comparison dashboard
models_data = {
    "FP32": {"accuracy": fp32_acc, "size_mb": fp32_size, "p50_ms": bench_fp32["p50"],
             "throughput": bench_fp32["throughput"]},
    "PTQ INT8": {"accuracy": ptq_acc, "size_mb": ptq_size, "p50_ms": bench_ptq["p50"],
                 "throughput": bench_ptq["throughput"]},
    "QAT INT8": {"accuracy": qat_acc, "size_mb": qat_size, "p50_ms": bench_qat["p50"],
                 "throughput": bench_qat["throughput"]},
}

labels = list(models_data.keys())
colors = ["steelblue", "darkorange", "firebrick"]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Quantization Benchmark Dashboard: FP32 vs PTQ vs QAT', fontsize=13, fontweight='bold')

# Accuracy
ax = axes[0, 0]
accs = [models_data[l]["accuracy"] for l in labels]
bars = ax.bar(labels, accs, color=colors, alpha=0.85)
ax.set_title('Test Accuracy')
ax.set_ylabel('Accuracy')
ax.set_ylim(max(0, min(accs) - 0.05), 1.05)
for bar, val in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'{val:.4f}', ha='center', fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

# Model size
ax = axes[0, 1]
sizes = [models_data[l]["size_mb"] for l in labels]
bars = ax.bar(labels, sizes, color=colors, alpha=0.85)
ax.set_title('Model Size')
ax.set_ylabel('Size (MB)')
for bar, val in zip(bars, sizes):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.02,
            f'{val:.3f}', ha='center', fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

# Latency
ax = axes[1, 0]
latencies = [models_data[l]["p50_ms"] for l in labels]
bars = ax.bar(labels, latencies, color=colors, alpha=0.85)
ax.set_title('Inference Latency (p50)')
ax.set_ylabel('Latency (ms)')
for bar, val in zip(bars, latencies):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.02,
            f'{val:.3f}', ha='center', fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

# Throughput
ax = axes[1, 1]
throughputs = [models_data[l]["throughput"] for l in labels]
bars = ax.bar(labels, throughputs, color=colors, alpha=0.85)
ax.set_title('Throughput')
ax.set_ylabel('Samples/second')
for bar, val in zip(bars, throughputs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.02,
            f'{val:.0f}', ha='center', fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Summary table
print("\n" + "=" * 80)
print("FINAL BENCHMARK SUMMARY")
print("=" * 80)
print(f"{'Metric':<25} | {'FP32':>12} | {'PTQ INT8':>12} | {'QAT INT8':>12}")
print("-" * 70)
print(f"{'Accuracy':<25} | {fp32_acc:>12.4f} | {ptq_acc:>12.4f} | {qat_acc:>12.4f}")
print(f"{'Accuracy Drop':<25} | {'—':>12} | {fp32_acc-ptq_acc:>12.4f} | {fp32_acc-qat_acc:>12.4f}")
print(f"{'Model Size (MB)':<25} | {fp32_size:>12.3f} | {ptq_size:>12.3f} | {qat_size:>12.3f}")
print(f"{'Compression Ratio':<25} | {'1.00x':>12} | {fp32_size/ptq_size:>11.2f}x | {fp32_size/qat_size:>11.2f}x")
print(f"{'Latency p50 (ms)':<25} | {bench_fp32['p50']:>12.3f} | {bench_ptq['p50']:>12.3f} | {bench_qat['p50']:>12.3f}")
print(f"{'Speedup':<25} | {'1.00x':>12} | {bench_fp32['p50']/bench_ptq['p50']:>11.2f}x | {bench_fp32['p50']/bench_qat['p50']:>11.2f}x")
print(f"{'Throughput (s/s)':<25} | {bench_fp32['throughput']:>12.0f} | {bench_ptq['throughput']:>12.0f} | {bench_qat['throughput']:>12.0f}")
print("=" * 80)

## 9. SQNR Analysis

In [ ]:
# Compute SQNR for each layer's weights
def compute_sqnr(fp32_w, quant_w):
    fp32_np = fp32_w.flatten().numpy() if hasattr(fp32_w, 'numpy') else fp32_w
    quant_np = quant_w.flatten().numpy() if hasattr(quant_w, 'numpy') else quant_w
    signal_power = np.mean(fp32_np ** 2)
    noise_power = np.mean((fp32_np - quant_np) ** 2)
    if noise_power < 1e-15:
        return float('inf')
    return 10 * np.log10(signal_power / noise_power)

sqnr_results = {}
for name in common:
    if name in fp32_weights and name in ptq_weights:
        sqnr = compute_sqnr(fp32_weights[name], ptq_weights[name])
        sqnr_results[name] = sqnr
        print(f"Layer {name}: SQNR = {sqnr:.2f} dB")

if sqnr_results:
    fig, ax = plt.subplots(figsize=(10, 4))
    names = list(sqnr_results.keys())
    sqnr_vals = list(sqnr_results.values())
    colors_sqnr = ['green' if v > 40 else 'orange' if v > 30 else 'red' for v in sqnr_vals]
    bars = ax.bar(names, sqnr_vals, color=colors_sqnr, alpha=0.8)
    ax.axhline(40, color='green', linestyle='--', label='Good (>40 dB)')
    ax.axhline(30, color='orange', linestyle='--', label='Acceptable (>30 dB)')
    ax.set_ylabel('SQNR (dB)')
    ax.set_title('SQNR per Layer: FP32 vs PTQ INT8 Weights')
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    for bar, val in zip(bars, sqnr_vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{val:.1f}', ha='center', fontsize=10)
    plt.tight_layout()
    plt.show()

## 10. Key Takeaways

| Insight | Detail |
|---------|--------|
| **PTQ is fast** | No retraining needed — just calibration data (~100 samples) |
| **QAT recovers accuracy** | Adapts weights during training to compensate for quantization error |
| **Both give ~4x compression** | INT8 stores 1 byte vs FP32's 4 bytes per weight |
| **Calibration method matters** | MSE/KL often gives lower quantization error than Min-Max for outlier-heavy distributions |
| **SQNR measures quality** | >40 dB per layer is the target for good quantization |
| **STE enables QAT** | The Straight-Through Estimator allows gradients to flow through rounding |
| **BN folding reduces ops** | Fold Conv+BN into a single Conv before quantization |
| **Mixed precision is a trade-off** | Keep sensitive layers at FP16, quantize the rest to INT8 |

See `docs/concepts.md` for detailed mermaid diagrams and theory.